# Regresión lineal en `adult.csv`

Este notebook resuelve un **modelo de regresión lineal** sobre el dataset `adult.csv`:

- Primero se divide en **train/test**.
- Después se ajustan (`fit`) imputadores, escalado y one-hot **solo con train**.
- Finalmente se transforma test y se evalúa el modelo.

**Variable objetivo elegida:** `hours-per-week` (numérica, adecuada para regresión).

## 1. Librerías

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import  mean_squared_error, r2_score


## 2. Carga del dataset y limpieza básica

En `adult.csv` los valores ausentes suelen venir como `?`. Se convierten a `NaN` y se eliminan filas incompletas.

In [7]:
df = pd.read_csv("adult.csv")
print(df.shape)
df = df.replace("?", np.nan)
df = df.dropna() #eliminamos filas que contienen NaN

df.head()
df.shape 

(32561, 15)


(30162, 15)

## 3. Definir X e y

Para regresión lineal necesitamos un objetivo numérico: `hours-per-week`.

In [11]:
y = df["hours.per.week"]
X = df.drop("hours.per.week", axis=1)

X.shape, y.shape

((30162, 14), (30162,))

## 4. División train/test

Importante: **se divide antes de transformar** para evitar fuga de información del test.

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

X_train.shape, X_test.shape

((24129, 14), (6033, 14))

## 5. Separar columnas numéricas y categóricas

In [13]:
cols_num = X_train.select_dtypes(include=["int64", "float64"]).columns
cols_cat = X_train.select_dtypes(include=["object"]).columns

print("Numéricas:", list(cols_num))
print("Categóricas:", list(cols_cat))

Numéricas: ['age', 'fnlwgt', 'education.num', 'capital.gain', 'capital.loss']
Categóricas: ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race', 'sex', 'native.country', 'income']


## 6. Transformaciones manuales

### 6.1 Numéricas: imputación + escalado
- `SimpleImputer(median)` y `StandardScaler`

### 6.2 Categóricas: imputación + One-Hot
- `SimpleImputer(most_frequent)` y `OneHotEncoder(handle_unknown='ignore')`


In [14]:
# ===== Numéricas =====
imputer_num = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train_num = imputer_num.fit_transform(X_train[cols_num])
X_test_num  = imputer_num.transform(X_test[cols_num])

X_train_num = scaler.fit_transform(X_train_num)
X_test_num  = scaler.transform(X_test_num)

# ===== Categóricas =====
imputer_cat = SimpleImputer(strategy="most_frequent")
# sparse_output=False para obtener un array denso (más sencillo para np.hstack)
ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

X_train_cat = imputer_cat.fit_transform(X_train[cols_cat])
X_test_cat  = imputer_cat.transform(X_test[cols_cat])

X_train_cat = ohe.fit_transform(X_train_cat)
X_test_cat  = ohe.transform(X_test_cat)

# ===== Unir =====
X_train_final = np.hstack([X_train_num, X_train_cat])
X_test_final  = np.hstack([X_test_num, X_test_cat])

X_train_final.shape, X_test_final.shape

((24129, 105), (6033, 105))

## 7. Entrenamiento del modelo (LinearRegression)

In [15]:
lr = LinearRegression()
lr.fit(X_train_final, y_train)

lr

LinearRegression()

## 8. Predicción y evaluación

- **MAE**: error medio (en horas/semana).
- **RMSE**: penaliza más errores grandes.
- **R²**: proporción de varianza explicada (suele ser moderado/bajo en este dataset).

In [16]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y_pred = lr.predict(X_test_final)

# Métricas
mae = mean_absolute_error(y_test, y_pred)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

r2 = r2_score(y_test, y_pred)

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²: {r2:.3f}")


MAE: 7.42
RMSE: 10.77
R²: 0.191


In [17]:
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error

rmse = root_mean_squared_error(y_test, y_pred)
